In [ ]:
import numpy as np
from tensorflow.keras.datasets import reuters
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, RepeatVector, TimeDistributed, Embedding, Dropout, Bidirectional
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Parameters
vocab_size = 10000
max_seq_length = 100  # Maximum length of sequences
embedding_dim = 100   # Size of word embeddings
latent_dim = 64     # Size of the latent space
rnn_cells_first = 256  # Number of cells in the first RNN layer
rnn_cells_second = 128  # Number of cells in the second RNN layer
epochs = 100
batch_size = 256

# Load Reuters dataset
(x_train, _), (x_test, _) = reuters.load_data(num_words=vocab_size)
x_train = pad_sequences(x_train, maxlen=max_seq_length, padding='post')
x_test = pad_sequences(x_test, maxlen=max_seq_length, padding='post')


# Encoder
input_text = Input(shape=(max_seq_length,))
x = Embedding(vocab_size, embedding_dim)(input_text)
x = Bidirectional(LSTM(rnn_cells_first, return_sequences=True))(x)
x = Dropout(0.15)(x)
x = Bidirectional(LSTM(rnn_cells_second))(x)
latent_repr = Dense(latent_dim, activation='relu')(x)
encoder_model = Model(input_text, latent_repr)
encoder_model.summary()

In [ ]:

# Decoder
decoder_input = Input(shape=(latent_dim,))
decoder_x = Dense(max_seq_length * embedding_dim, activation='relu')(decoder_input)
decoder_x = RepeatVector(max_seq_length)(decoder_x)
decoder_x = LSTM(rnn_cells_second, return_sequences=True)(decoder_x)
decoder_x = Dropout(0.15)(decoder_x)
decoder_x = LSTM(rnn_cells_first, return_sequences=True)(decoder_x)
decoder_output = TimeDistributed(Dense(vocab_size, activation='softmax'))(decoder_x)
decoder_model = Model(decoder_input, decoder_output)
decoder_model.summary()

In [ ]:

# Autoencoder
autoencoder = Model(input_text, decoder_model(encoder_model(input_text)))
autoencoder.compile(optimizer=Adam(learning_rate=0.0005), loss='sparse_categorical_crossentropy')
autoencoder.summary()

In [ ]:

from tensorflow.keras.callbacks import LearningRateScheduler
import tensorflow.keras.backend as K

def custom_lr_scheduler(epoch, lr):
    # Decrease learning rate by 0.1 factor every 5 epochs
    if epoch % 5 == 0 and epoch != 0:
        lr = lr * 0.1
    return lr

# Define the callback
lr_scheduler = LearningRateScheduler(custom_lr_scheduler)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
autoencoder.fit(x_train, x_train, epochs=epochs, batch_size=batch_size, validation_data=(x_test, x_test), callbacks=[early_stopping, lr_scheduler])


In [ ]:
# Encode the input text using the encoder model
encoded_texts = encoder_model.predict(x_test[:10])

# Decode the encoded texts using the decoder model
# This gives you a probability distribution for each word in the sequence
prob_distributions = decoder_model.predict(encoded_texts)


In [ ]:
def sample(preds, temperature=1.0):
    # Convert to array and prevent numerical issues with very small numbers
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature  # Adjust by temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)  # Softmax
    probas = np.random.multinomial(1, preds, 1)  # Sample from the softmax distribution
    return np.argmax(probas)


In [ ]:
word_index = reuters.get_word_index()
reverse_word_index = {value: key for (key, value) in word_index.items()}
def decode_sequence_with_sampling(prob_distributions, temperature=1.0):
    return ' '.join([reverse_word_index.get(sample(probs, temperature) - 3, '?') for probs in prob_distributions])

# Example usage with a specified temperature
temperature = 1  # Adjust this value to change randomness
decoded_texts = [decode_sequence_with_sampling(seq, temperature) for seq in prob_distributions]
for text in decoded_texts:
    print('\n')
    print(f'Decoded: {text}')
    print('\n')
